In [58]:
# Reload the modules
%load_ext autoreload
%autoreload 2

# Environment Imports
import torch, torchaudio, numpy as np, matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from sylber.model.sylber import Sylber
from sylber.utils.segment_utils import get_segment
import importlib, inspect
from pathlib import Path
from sklearn.manifold import TSNE
import ipywidgets as w

import params as _params
import tuner as _tuner
import experiments.cutters as cutters
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [59]:
wav_path = Path("../samples/audio_files/data/09/behav_sub-09_run-01_stim-35_ri ri ri ri ri ri ri ri ri.wav")
print("exists:", wav_path.exists())
print("absolute:", wav_path.resolve())

# Load audio and resample to 16 kHz mono
wav, sr = torchaudio.load(wav_path)
wav = torchaudio.functional.resample(wav, sr, 16_000).mean(0, keepdim=True)

# Fetch embeddings
model = Sylber(segment_online=False).eval()
with torch.no_grad():
    states = model.speech_model(wav).last_hidden_state[0].detach().cpu().numpy()

print('Embedding shape:', states.shape)

exists: True
absolute: /Users/sammichaels/Summer 2025 USRA/sylber/samples/audio_files/data/09/behav_sub-09_run-01_stim-35_ri ri ri ri ri ri ri ri ri.wav
Embedding shape: (224, 768)


In [60]:
# grab only callables that take (states, …) and return a list/tuple
cutters = importlib.reload(cutters)
EXCLUDE = {"get_segment"}

CUT_FUNCS = {
    name: fn
    for name, fn in inspect.getmembers(cutters, inspect.isfunction)
    if fn.__code__.co_varnames[:1] == ('states',) and name not in EXCLUDE
}

print("Loaded cutters:", list(CUT_FUNCS))

Loaded cutters: ['bocpd_offline', 'energy_drop', 'foote_novelty', 'sylber_greedy']


In [76]:
# ------------------------------------------------------------------
# Rename WAV files: " "  →  "_"
# ------------------------------------------------------------------
from pathlib import Path

AUDIO_ROOT = Path("../samples/audio_files/data/09")      # adjust if needed
count = 0

for wav in AUDIO_ROOT.rglob("*.wav"):                    # walk sub-dirs too
    new_name = wav.name.replace(" ", "_")
    if new_name == wav.name:
        continue                                         # already OK
    new_path = wav.with_name(new_name)
    print(f"{wav.name}  →  {new_name}")
    wav.rename(new_path)
    count += 1

print(f"\nRenamed {count} WAV file(s).")

behav_sub-09_run-02_stim-38_po ta to be ne fit pe ri od.wav  →  behav_sub-09_run-02_stim-38_po_ta_to_be_ne_fit_pe_ri_od.wav
behav_sub-09_run-01_stim-03_be ne fit po ta to pe ri od.wav  →  behav_sub-09_run-01_stim-03_be_ne_fit_po_ta_to_pe_ri_od.wav
behav_sub-09_run-02_stim-28_pe ri od be ne fit po ta to.wav  →  behav_sub-09_run-02_stim-28_pe_ri_od_be_ne_fit_po_ta_to.wav
behav_sub-09_run-01_stim-08_po ta to pe ri od be ne fit.wav  →  behav_sub-09_run-01_stim-08_po_ta_to_pe_ri_od_be_ne_fit.wav
behav_sub-09_run-01_stim-14_po ta to be ne fit pe ri od.wav  →  behav_sub-09_run-01_stim-14_po_ta_to_be_ne_fit_pe_ri_od.wav
behav_sub-09_run-01_stim-40_ne ne ne ne ne ne ne ne ne.wav  →  behav_sub-09_run-01_stim-40_ne_ne_ne_ne_ne_ne_ne_ne_ne.wav
behav_sub-09_run-01_stim-19_po fit to ta ri od be ne pe.wav  →  behav_sub-09_run-01_stim-19_po_fit_to_ta_ri_od_be_ne_pe.wav
behav_sub-09_run-01_stim-38_po ta to be ne fit pe ri od.wav  →  behav_sub-09_run-01_stim-38_po_ta_to_be_ne_fit_pe_ri_od.wav
behav_sub-

In [80]:
# ------------------------------------------------------------------
# Build `clips`: use TextGrids as the source of truth
# (handles “_” ↔︎ space mismatch between stems)
# ------------------------------------------------------------------
VAL_DIR = Path("../samples/audio_files/data")      # root with 00/01/03/07/09
TG_ROOT = VAL_DIR / "TextGrid"                     # mirrors the same sub-folders

clips = []   # (wav_path, ref_times, wav, states)

for tg_path in TG_ROOT.rglob("*.TextGrid"):
    rel   = tg_path.relative_to(TG_ROOT)           # e.g. 09/behav_sub-…
    stem  = tg_path.stem.replace(" ", "_")         # swap space → _
    wav_path = VAL_DIR / rel.parent / f"{stem}.wav"   # …/data/09/<stem>.wav

    if not wav_path.exists():
        print("WAV not found for", tg_path.name)
        continue

    # --- load audio & embeddings ----------------------------------
    wav, sr = torchaudio.load(wav_path)
    wav = torchaudio.functional.resample(wav, sr, 16_000).mean(0, keepdim=True)
    with torch.no_grad():
        states = (
            model.speech_model(wav)
            .last_hidden_state[0]
            .detach()
            .cpu()
            .numpy()
        )

    # --- load ground-truth boundaries -----------------------------
    import praatio.textgrid as ptg
    tg_obj = ptg.openTextgrid(tg_path, includeEmptyIntervals=True)
    tier   = tg_obj.getTier("syll")                # access by tier name
    ref_times = np.array([interval.start for interval in tier.entries[1:]])

    clips.append((wav_path, ref_times, wav, states))

print(f"Collected {len(clips)} annotated clips")

Collected 42 annotated clips


In [83]:
importlib.reload(_params)
_tuner = importlib.reload(_tuner)

grid_results = []

for name, fn in inspect.getmembers(cutters, inspect.isfunction):
    if name not in _params.SEARCH_SPACE:
        continue
    print(f"→ tuning {name}")
    df = _tuner.evaluate_algo(name, fn, _params.SEARCH_SPACE[name], clips)
    display(df.head(7))  # show top 7 configs
    grid_results.append(df)

best_overall = (
    pd.concat(grid_results)
      .sort_values("f_mean", ascending=False)
      .reset_index(drop=True)
)

best_overall.head(10)

→ tuning bocpd_offline


KeyboardInterrupt: 

In [ ]:
all_segments = {}
for name, fn in CUT_FUNCS.items():
    segs = fn(states)
    all_segments[name] = segs
    print(f"{name:15} → {len(segs)} segments")

In [ ]:
dist = np.diag(cdist(states, states, "cosine"), k=1)
plt.figure(figsize=(12, 3))
plt.plot(dist, lw=0.7, label="Δ-cosine")
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for idx, (name, segs) in enumerate(all_segments.items()):
    for s, _ in segs:
        plt.axvline(s, color=colors[idx % len(colors)], ls="--", lw=0.8)
    # dummy line for legend
    plt.plot([], [], c=colors[idx % len(colors)], label=name)

plt.legend()
plt.title("Cut positions from each algorithm")
plt.xlabel("Frame index (≈20 ms per frame)")
plt.show()

In [ ]:
def demo_foote(L=12, peak=1.5):
    segs = cutters.foote_novelty(states, ker_win=L, peak_thr=peak)
    plt.figure(figsize=(10, 2))
    plt.plot(dist, lw=0.7)
    for s, _ in segs:
        plt.axvline(s, color="r", ls="--", lw=0.8)
    plt.title(f"Foote L={L}, peak_thr={peak}, segments={len(segs)}")
    plt.show()

w.interact(
    demo_foote,
    L=w.IntSlider(value=12, min=4, max=30, step=2),
    peak=w.FloatSlider(value=1.5, min=0.5, max=3, step=0.1),
);